In [1]:
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

# ============================================================
# PATHS — sesuaikan dengan struktur folder kamu
# ============================================================
TRAIN_CSV        = 'youtube_enriched_cluster.csv'   # CSV training asli
NEW_CSV          = 'new_videos.csv'                  # CSV hasil collect tadi
TRAIN_THUMB_DIR  = 'thumbnails'                      # folder thumbnail training
NEW_THUMB_DIR    = 'thumbnails_new'                  # folder thumbnail baru
OUTPUT_CSV       = 'new_videos_labeled.csv'          # output akhir
MODELS_DIR       = 'saved_models'                    # folder untuk simpan pkl
# ============================================================

os.makedirs(MODELS_DIR, exist_ok=True)

In [2]:
METADATA_COLS = [
    'view_rate', 'like_count', 'comment_count', 'video_age_days',
    'like_rate', 'comment_rate', 'engagement_rate',
    'subscriber_count', 'log_view_count', 'log_like_count',
    'log_comment_count', 'log_subscriber_count'
]

# --- Load CNN model ---
print("🔧 Loading EfficientNetB0...")
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
gap_output = GlobalAveragePooling2D()(base_model.output)
cnn_model  = Model(inputs=base_model.input, outputs=gap_output)


def extract_cnn_features(df, thumb_dir):
    """Extract CNN features from thumbnails."""
    images = []
    valid_idx = []
    for i, row in df.iterrows():
        # support both 'thumbnails/id.jpg' and just 'id'
        vid_id = os.path.splitext(os.path.basename(str(row.get('thumbnail_path', ''))))[0]
        path   = os.path.join(thumb_dir, f'{vid_id}.jpg')
        if not os.path.exists(path):
            # try underscore prefix (some folders use _id.jpg)
            path = os.path.join(thumb_dir, f'_{vid_id}.jpg')
        if not os.path.exists(path):
            print(f"   ⚠️  Thumbnail not found: {vid_id}, skipping row {i}")
            continue
        img = load_img(path, target_size=(224, 224))
        images.append(preprocess_input(img_to_array(img)))
        valid_idx.append(i)

    if not images:
        raise FileNotFoundError(f"No thumbnails found in '{thumb_dir}'. Check the path.")

    features = cnn_model.predict(np.array(images), verbose=1, batch_size=16)
    return features, valid_idx


# ============================================================
# STEP 1 — Re-fit Scaler, OHE, KMeans on training data
# ============================================================
print("\n📂 Loading training data...")
train_df = pd.read_csv(TRAIN_CSV)

# Compute derived features if missing
for col in ['log_view_count', 'log_like_count', 'log_comment_count', 'log_subscriber_count']:
    src = col.replace('log_', '')
    if col not in train_df.columns:
        train_df[col] = np.log1p(train_df[src])

for col in ['like_rate', 'comment_rate', 'engagement_rate']:
    if col not in train_df.columns:
        if col == 'like_rate':
            train_df[col] = train_df['like_count'] / train_df['view_count']
        elif col == 'comment_rate':
            train_df[col] = train_df['comment_count'] / train_df['view_count']
        elif col == 'engagement_rate':
            train_df[col] = (train_df['like_count'] + train_df['comment_count']) / train_df['view_count']

print("🖼️  Extracting CNN features from training thumbnails...")
X_cnn_train, train_valid_idx = extract_cnn_features(train_df, TRAIN_THUMB_DIR)
train_df = train_df.loc[train_valid_idx].reset_index(drop=True)

print("⚙️  Fitting OHE + Scaler + KMeans on training data...")
ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')
cat_train = ohe.fit_transform(train_df[['Category']])

meta_train = train_df[METADATA_COLS].values
scaler     = StandardScaler()
meta_scaled = scaler.fit_transform(np.concatenate([meta_train,
                                    np.zeros((len(meta_train), cat_train.shape[1]))], axis=1))
# scale full combined metadata+ohe together
X_meta_train = np.concatenate([meta_train, cat_train], axis=1)
X_meta_scaled = scaler.fit_transform(X_meta_train)   # re-fit properly
X_all_train   = np.concatenate([X_cnn_train, X_meta_scaled], axis=1)

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_all_train)

# Save fitted objects
joblib.dump(scaler, f'{MODELS_DIR}/scaler.pkl')
joblib.dump(kmeans, f'{MODELS_DIR}/kmeans.pkl')
joblib.dump(ohe,    f'{MODELS_DIR}/ohe.pkl')
print(f"✅ Scaler, KMeans, OHE saved to '{MODELS_DIR}/'")

# Verify cluster means match original
train_df['cluster_label_refit'] = kmeans.labels_
print("\n📊 Cluster means (refit) — should match your original:")
print(train_df.groupby('cluster_label_refit')[['view_rate', 'like_rate', 'engagement_rate']]
      .mean().sort_values('view_rate', ascending=False))


🔧 Loading EfficientNetB0...

📂 Loading training data...
🖼️  Extracting CNN features from training thumbnails...
65/65 [==============================] - 5s 23ms/step
⚙️  Fitting OHE + Scaler + KMeans on training data...


C:\Users\LENOVO\anaconda3\envs\tf-gpu-clean\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\LENOVO\anaconda3\envs\tf-gpu-clean\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


✅ Scaler, KMeans, OHE saved to 'saved_models/'

📊 Cluster means (refit) — should match your original:
                        view_rate  like_rate  engagement_rate
cluster_label_refit                                          
2                    12016.853265   0.029290         0.031104
1                     5251.256054   0.024102         0.028496
0                     3833.517541   0.046249         0.047751

📂 Loading new videos...
🖼️  Extracting CNN features from new thumbnails...
101/101 [==============================] - 2s 22ms/step
🔮 Assigning cluster labels...

✅ Done! 1609 videos labeled → 'new_videos_labeled.csv'

📊 Cluster distribution in new data:
cluster_label
0    964
1    531
2    114
Name: count, dtype: int64

📊 Cluster means (new data):
               view_rate  like_rate  engagement_rate
cluster_label                                       
1               0.317972   0.026464         0.031477
0               0.108688   0.034829         0.036322
2               0.074811 

   view_ratio  view_rate
0    0.029119   0.029119
1    0.004957   0.004957
2    0.010031   0.010031
3    0.325420   0.325420
4    0.009416   0.009416
     view_rate
0  1350.396166
1   903.414634
2  2150.603704
3  6801.700000
4  2080.943396


In [4]:
new_df['view_rate'] = new_df.get('view_ratio', new_df.get('view_rate', 0))

In [5]:
print(new_df[['view_ratio', 'view_rate']].head())
print(train_df[['view_rate']].head())

   view_ratio  view_rate
0    0.029119   0.029119
1    0.004957   0.004957
2    0.010031   0.010031
3    0.325420   0.325420
4    0.009416   0.009416
     view_rate
0  1350.396166
1   903.414634
2  2150.603704
3  6801.700000
4  2080.943396


In [6]:
print(train_df[['view_count', 'subscriber_count', 'view_rate', 'view_ratio']].head())

   view_count  subscriber_count    view_rate  view_ratio
0      422674           3010000  1350.396166    0.140423
1       74080            516000   903.414634    0.143566
2      580663           2540000  2150.603704    0.228607
3      544136            832000  6801.700000    0.654010
4      110290           2730000  2080.943396    0.040399


In [7]:
print(train_df[['view_count', 'video_age_days', 'view_rate']].head())
print((train_df['view_count'] / train_df['video_age_days']).head())

   view_count  video_age_days    view_rate
0      422674             313  1350.396166
1       74080              82   903.414634
2      580663             270  2150.603704
3      544136              80  6801.700000
4      110290              53  2080.943396
0    1350.396166
1     903.414634
2    2150.603704
3    6801.700000
4    2080.943396
dtype: float64


In [8]:
new_df['view_rate'] = new_df['view_count'] / new_df['video_age_days']

In [9]:
# ============================================================
# STEP 2 — Compute derived features for new data
# ============================================================
print("\n📂 Loading new videos...")
new_df = pd.read_csv(NEW_CSV)

# video_age_days from published_at
if 'video_age_days' not in new_df.columns:
    new_df['published_at'] = pd.to_datetime(new_df['published_at'], utc=True)
    new_df['video_age_days'] = (pd.Timestamp.utcnow() - new_df['published_at']).dt.days

# log features
for col in ['log_view_count', 'log_like_count', 'log_comment_count', 'log_subscriber_count']:
    src = col.replace('log_', '')
    new_df[col] = np.log1p(new_df[src])

# rate features
new_df['like_rate']       = new_df['like_count'] / new_df['view_count'].replace(0, np.nan)
new_df['comment_rate']    = new_df['comment_count'] / new_df['view_count'].replace(0, np.nan)
new_df['engagement_rate'] = (new_df['like_count'] + new_df['comment_count']) / new_df['view_count'].replace(0, np.nan)
new_df['view_rate']       = new_df.get('view_ratio', new_df.get('view_rate', 0))
new_df.fillna(0, inplace=True)

# ============================================================
# STEP 3 — Extract CNN features for new thumbnails
# ============================================================
print("🖼️  Extracting CNN features from new thumbnails...")
X_cnn_new, new_valid_idx = extract_cnn_features(new_df, NEW_THUMB_DIR)
new_df = new_df.loc[new_valid_idx].reset_index(drop=True)


📂 Loading new videos...
🖼️  Extracting CNN features from new thumbnails...
101/101 [==============================] - 2s 20ms/step
🔮 Assigning cluster labels...

✅ Done! 1609 videos labeled → 'new_videos_labeled.csv'

📊 Cluster distribution in new data:
cluster_label
0    964
1    531
2    114
Name: count, dtype: int64

📊 Cluster means (new data):
               view_rate  like_rate  engagement_rate
cluster_label                                       
1               0.317972   0.026464         0.031477
0               0.108688   0.034829         0.036322
2               0.074811   0.023856         0.025247


In [10]:
print(new_df[['view_count', 'video_age_days', 'view_rate']].head())

   view_count  video_age_days  view_rate
0       73961               6   0.029119
1       12591              15   0.004957
2       25480              15   0.010031
3      826566              20   0.325420
4       23916              22   0.009416


In [11]:
new_df['view_rate'] = new_df['view_count'] / new_df['video_age_days']
print(new_df[['view_count', 'video_age_days', 'view_rate']].head())

   view_count  video_age_days     view_rate
0       73961               6  12326.833333
1       12591              15    839.400000
2       25480              15   1698.666667
3      826566              20  41328.300000
4       23916              22   1087.090909


In [12]:
# ============================================================
# STEP 4 — Transform and predict cluster labels
# ============================================================
print("🔮 Assigning cluster labels...")
cat_new = ohe.transform(new_df[['category']].rename(columns={'category': 'Category'}))
meta_new     = new_df[METADATA_COLS].values
X_meta_new   = np.concatenate([meta_new, cat_new], axis=1)
X_meta_new_s = scaler.transform(X_meta_new)          # use fitted Scaler
X_all_new    = np.concatenate([X_cnn_new, X_meta_new_s], axis=1)

new_df['cluster_label'] = kmeans.predict(X_all_new)  # use fitted KMeans

# ============================================================
# STEP 5 — Save output
# ============================================================
new_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Done! {len(new_df)} videos labeled → '{OUTPUT_CSV}'")
print("\n📊 Cluster distribution in new data:")
print(new_df['cluster_label'].value_counts().sort_index())
print("\n📊 Cluster means (new data):")
print(new_df.groupby('cluster_label')[['view_rate', 'like_rate', 'engagement_rate']]
      .mean().sort_values('view_rate', ascending=False))

🔮 Assigning cluster labels...


ValueError: Input X contains infinity or a value too large for dtype('float64').